# NB-03 · Test-set lock

Owner: F (W6 Evaluation). Checks the train/test split for leakage, adds the 105 hand-written cases, builds the held-out, production-like and adversarial test files in `data/test/`, and writes `LOCK.md` with a SHA-256 hash of every file.

Run once, top to bottom. Only `data/test/LOCK.md` is committed; the data files stay off GitHub.

In [ ]:
# NB-03 · Test-set lock · Owner: F (W6 Evaluation)
# Checks Shravan's train/test split for leakage, adds the hand-written cases,
# builds the held-out, production-like and adversarial test files, and writes
# LOCK.md with a SHA-256 hash of every file. Run once, top to bottom.

import csv, hashlib, json, math, random, re, sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

# ---------------------------------------------------------------- settings
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # repo root
DATA = ROOT / "data"                                  # change if your data folder is elsewhere
POLICY = ROOT / "tests" / "fixtures" / "config" / "policy.yaml"
TRAIN = DATA / "synthetic" / "train.jsonl"
TEST = DATA / "synthetic" / "test.jsonl"
PROTECTED = DATA / "protected"
HAND_TXT = [PROTECTED / "john.txt", PROTECTED / "abi.txt", PROTECTED / "saif.txt"]
HAND_LABELS = PROTECTED / "labels.csv"
EDGE = PROTECTED / "edge_cases.jsonl"                # policy v1.0 edge cases, triggers and look-alikes
ADV_SEALED = PROTECTED / "adversarial_sealed.jsonl"  # if missing, adversarial records in test.jsonl are used
OUT = DATA / "test"

SEED = 894
NEAR_DUP = 0.90          # text similarity at or above this counts as near-identical
PROD_ROUTINE_SHARE = 0.90
HOLDBACK = False         # True: hold back a third of the adversarial set for F (open decision)
HOLDBACK_SHARE = 1 / 3

TRIGGERS = ["SW-BIN/bin_injury", "AN-DEAD/private_property", "AN-DEAD/near_school", "AN-INJ/aggressive",
            "AN-INJ/in_traffic", "AN-INJ/dog_or_cat", "RD-POT/swerving", "RD-POT/vehicle_damage",
            "RD-POT/cyclists_at_risk", "RD-POT/deep", "RD-POT/hospital_route", "TR-TREE/limb_down",
            "TR-TREE/leaning_home", "TR-TREE/hospital_route", "PS-PROP/tenant_dispute",
            "PS-PROP/no_heat_or_water", "PS-PROP/structural_danger", "PS-PROP/vulnerable_residents",
            "IL-DUMP/hazardous", "IL-DUMP/near_school"]   # policy v1.0, category trigger table

# Fallback lists, used only if policy.yaml cannot be read or is still in the v0.1 shape
ROUTE_CODES = {"SW-BIN", "SW-MISS", "AN-DEAD", "AN-INJ", "RD-POT",
               "TR-TREE", "PS-PROP", "IL-DUMP", "EMERG", "OOS"}
OUTCOMES = {"create_ticket", "ask_clarification", "link_duplicate", "escalate_human",
            "redirect_911", "handoff_oos", "handle_each", "refuse_and_continue",
            "handoff_language", "report_status"}
policy_source = "built-in lists (policy.yaml not found or still v0.1)"
try:                                      # the codes and outcomes come from the policy, not this notebook
    sys.path.insert(0, str(ROOT))
    try:
        from src.config import policy as load_policy
        pol = load_policy()
        where = "src.config.policy()"
    except Exception:
        import yaml
        pol = yaml.safe_load(open(POLICY, encoding="utf-8"))
        where = str(POLICY.relative_to(ROOT))
    if pol.get("route_codes") and pol.get("outcomes"):
        ROUTE_CODES = set(pol["route_codes"])
        OUTCOMES = set(pol["outcomes"])
        policy_source = f"{where}, version {pol.get('version', 'unknown')}"
except Exception:
    pass

REQUIRED = ["id", "message", "service_code", "expected_action", "rule_id", "hazard_flag"]

GROUP_KEY = "template_id"  # the split group: the 3 paraphrases of one scenario share it

# ---------------------------------------------------------------- helpers
problems = []   # anything here stops the lock

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def to_bool(v):
    return str(v).strip().lower() in {"true", "1", "yes", "y"}

def norm(text):
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()

def is_routine(r):
    return r["expected_action"] == "create_ticket" and not r["hazard_flag"]

def wilson_low(k, n, z=1.96):
    if n == 0:
        return float("nan")
    p = k / n
    return (p + z*z/(2*n) - z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))) / (1 + z*z/n)

def validate(rows, name):
    for r in rows:
        missing = [k for k in REQUIRED if r.get(k) in (None, "")]
        if missing:
            problems.append(f"{name} {r.get('id')}: missing {missing}")
        if r.get("service_code") not in ROUTE_CODES:
            problems.append(f"{name} {r.get('id')}: unknown code {r.get('service_code')}")
        if r.get("expected_action") not in OUTCOMES:
            problems.append(f"{name} {r.get('id')}: unknown outcome {r.get('expected_action')}")
        r["hazard_flag"] = to_bool(r.get("hazard_flag"))
        r["lang"] = r.get("lang") or "en"

# ---------------------------------------------------------------- 1. load
train = load_jsonl(TRAIN)
test = load_jsonl(TEST)

def load_hand_written():
    labels = list(csv.DictReader(open(HAND_LABELS, encoding="utf-8")))
    rows = []
    for txt in HAND_TXT:
        author = txt.stem
        lines = [l.strip() for l in open(txt, encoding="utf-8") if l.strip()]
        mine = [l for l in labels if l.get("author", "").strip().lower() == author.lower()]
        if len(lines) != len(mine):
            problems.append(f"{txt.name}: {len(lines)} messages but {len(mine)} label rows for {author}")
            continue
        for msg, lab in zip(lines, mine):         # matched by order within each author
            r = {k: (v.strip() if isinstance(v, str) else v) for k, v in lab.items()}
            r["message"] = r.get("message") or msg
            for k in ("gold_fields", "omitted_fields", "secondary_codes"):
                if r.get(k):
                    r[k] = json.loads(r[k])
            for k in ("trigger_id", "ticket_ref"):
                r[k] = r.get(k) or None
            r.update(model="human", persona=author, channel=r.get("channel") or "chat")
            rows.append(r)
    return rows

hand = load_hand_written()
edge = load_jsonl(EDGE) if EDGE.exists() else []
for r in edge:                                   # placeholders left for review stop the lock
    if r.get("ticket_ref") == "CHECK" or "CHOOSE OUTCOME" in (r.get("review_note") or ""):
        problems.append(f"edge case {r['id']}: still has a review placeholder ({r.get('review_note')[:80]})")

if ADV_SEALED.exists():
    adversarial = load_jsonl(ADV_SEALED)
    adv_source = ADV_SEALED.name
else:
    adversarial = [r for r in test if str(r.get("difficulty", "")).lower() == "adversarial"]
    adv_source = "adversarial records in test.jsonl"
test_main = [r for r in test if str(r.get("difficulty", "")).lower() != "adversarial"]

# The template check only works if generated records carry their template (group).
for r in train + test:                     # older files called it "group"
    r.setdefault(GROUP_KEY, r.get("group"))
no_group = [r.get("id") for r in train + test if not r.get(GROUP_KEY)]
if no_group:
    problems.append(f"{len(no_group)} generated records have no '{GROUP_KEY}' (scenario template), so the "
                    f"template check cannot run, e.g. {no_group[:5]}")
if not adversarial:
    problems.append(f"No adversarial cases found: {ADV_SEALED.name} is missing and no record in "
                    f"test.jsonl has difficulty 'adversarial'")

for rows, name in [(train, "train"), (test_main, "test"), (hand, "hand-written"), (edge, "edge case"),
                   (adversarial, "adversarial")]:
    validate(rows, name)

# ---------------------------------------------------------------- 2. leakage checks
all_test = test_main + hand + edge + adversarial
checks = {}

ids = [r["id"] for r in train + all_test]
dupe_ids = [i for i, c in Counter(ids).items() if c > 1]
checks["No shared or repeated IDs"] = dupe_ids

train_groups = {r.get(GROUP_KEY) for r in train if r.get(GROUP_KEY)}
shared_groups = sorted({r[GROUP_KEY] for r in all_test if r.get(GROUP_KEY) in train_groups})
checks["No scenario template in both train and test"] = shared_groups

train_text = {norm(r["message"]): r["id"] for r in train}
identical = [(r["id"], train_text[norm(r["message"])]) for r in all_test if norm(r["message"]) in train_text]
checks["No identical message in train and test"] = identical

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit([r["message"] for r in train + all_test])
sim = cosine_similarity(vec.transform([r["message"] for r in all_test]),
                        vec.transform([r["message"] for r in train]))
from difflib import SequenceMatcher
near = []
for i, row in enumerate(sim):            # shortlist the 5 closest train messages, then compare letter by letter
    a = norm(all_test[i]["message"])
    for j in row.argsort()[-5:]:
        b = norm(train[j]["message"])
        score = SequenceMatcher(None, a, b).ratio()
        if a != b and score >= NEAR_DUP:
            near.append((all_test[i]["id"], train[j]["id"], round(score, 3)))
            break
checks[f"No near-identical message (similarity >= {NEAR_DUP})"] = near

# Near-identical generated cases are dropped from the test files and listed in
# near_duplicates_dropped.csv. A hand-written case close to train stops the lock:
# the fix is to remove the train message (Shravan's step 9), not the hand-written case.
near_name = f"No near-identical message (similarity >= {NEAR_DUP})"
hand_ids = {r["id"] for r in hand + edge}
near_hand = [h for h in near if h[0] in hand_ids]
dropped = {h[0] for h in near if h[0] not in hand_ids}
checks[near_name] = near_hand
test_main = [r for r in test_main if r["id"] not in dropped]
adversarial = [r for r in adversarial if r["id"] not in dropped]

for name, hits in checks.items():
    if hits:
        problems.append(f"{name}: {len(hits)} found, e.g. {hits[:5]}")

# ---------------------------------------------------------------- 3. build the test sets
rng = random.Random(SEED)
heldout = test_main + hand + edge

# Trigger coverage: every category trigger needs at least one must-escalate case and one
# look-alike in the held-out set, or its recall cannot be measured (policy v1.0).
def tkey(t):
    return (t or "").split("/")[-1]
coverage = []
for t in TRIGGERS:
    code, tid = t.split("/")
    mine = [r for r in heldout if tkey(r.get("trigger_id")) == tid and r.get("service_code") == code]
    must = [r for r in mine if r["expected_action"] == "escalate_human"]
    look = [r for r in mine if r["expected_action"] not in ("escalate_human", "redirect_911")]
    gen = lambda rows: sum(1 for r in rows if r.get("model") != "human")
    coverage.append((t, len(must), gen(must), len(look), gen(look)))
    if not must or not look:
        problems.append(f"Trigger {t} has {len(must)} must-escalate and {len(look)} look-alike cases in held-out")

routine = [r for r in heldout if is_routine(r)]
other = [r for r in heldout if not is_routine(r)]
n_other = min(len(other), int(len(routine) * (1 - PROD_ROUTINE_SHARE) / PROD_ROUTINE_SHARE))
n_routine = min(len(routine), round(n_other * PROD_ROUTINE_SHARE / (1 - PROD_ROUTINE_SHARE)))
prod = rng.sample(routine, n_routine) + rng.sample(other, n_other)
rng.shuffle(prod)

holdback = []
if HOLDBACK:
    by_type = {}
    for r in adversarial:
        by_type.setdefault(r.get("adversarial_type") or r.get("rule_id"), []).append(r)
    for rows in by_type.values():          # a third of each attack type
        rows = rows[:]
        rng.shuffle(rows)
        holdback += rows[:round(len(rows) * HOLDBACK_SHARE)]
    hold_ids = {r["id"] for r in holdback}
    adversarial_fix = [r for r in adversarial if r["id"] not in hold_ids]

if problems:
    print("LOCK STOPPED. Fix these and run again:\n")
    for p in problems:
        print(" -", p)
    raise RuntimeError("Lock stopped: fix the problems listed above, then run again.")

# ---------------------------------------------------------------- 4. write files and LOCK.md
OUT.mkdir(parents=True, exist_ok=True)
files = {"test_heldout.jsonl": heldout, "test_prod.jsonl": prod}
if HOLDBACK:
    files.update({"adversarial_for_fixing.jsonl": adversarial_fix,
                  "adversarial_holdback.jsonl": holdback})
else:
    files["test_adversarial.jsonl"] = adversarial
for name, rows in files.items():
    write_jsonl(OUT / name, rows)
with open(OUT / "near_duplicates_dropped.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["test_id", "closest_train_id", "similarity"])
    w.writerows(h for h in near if h[0] in dropped)

def table(rows, key):
    c = Counter(str(r.get(key)) for r in rows)
    return "\n".join(f"| {k} | {v} |" for k, v in sorted(c.items()))

hz = [r for r in heldout if r["hazard_flag"]]
now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
inputs = [TRAIN, TEST, HAND_LABELS, *HAND_TXT] + ([EDGE] if EDGE.exists() else []) + ([ADV_SEALED] if ADV_SEALED.exists() else [])

lines = [
    "# Test-set lock", "",
    f"Locked {now} · Owner: F (W6 Evaluation) · Seed {SEED}", "",
    f"Route codes and outcomes checked against: {policy_source}", "",
    "These files are frozen. Only the eval harness reads them. Nothing is tuned on them. "
    "The data files stay off GitHub; only this file is committed. Check any shared copy "
    "against the hashes below (`shasum -a 256 <file>`).", "",
    "## Locked files", "", "| File | Cases | SHA-256 |", "|---|---|---|",
    *[f"| {n} | {len(rows)} | `{sha256(OUT / n)}` |" for n, rows in files.items()], "",
    "## Inputs", "", "| File | SHA-256 |", "|---|---|",
    *[f"| {p.relative_to(ROOT)} | `{sha256(p)}` |" for p in inputs], "",
    "## Leakage checks (train vs every test file)", "",
    *[f"- [x] {name}" for name in checks],
    f"- {len(dropped)} generated test cases were near-identical to a train message and were "
    f"dropped (listed in near_duplicates_dropped.csv). No hand-written case was near-identical.", "",
    "## What is in each set", "",
    f"- **Held-out:** {len(test_main)} generated test cases, {len(hand)} hand-written cases and "
    f"{len(edge)} policy edge cases.",
    f"- **Production-like:** drawn from the held-out set, {n_routine} routine "
    f"(create_ticket, no hazard) and {n_other} other, about {PROD_ROUTINE_SHARE:.0%} routine.",
    f"- **Adversarial:** {len(adversarial)} cases from {adv_source}."
    + (f" {len(holdback)} held back for F, stratified by attack type; G7 is measured on those."
       if HOLDBACK else ""),
    f"- **Hazard cases in held-out (G1):** {len(hz)}. If every one is caught, the 95% Wilson "
    f"lower bound is {wilson_low(len(hz), len(hz)):.1%}; one miss gives recall "
    f"{(len(hz) - 1) / len(hz):.1%}." if hz else "- **Hazard cases in held-out (G1):** 0",
    "", "### Trigger coverage in held-out (all cases / generated only)", "",
    "| Trigger | Must escalate | Look-alike |", "|---|---|---|",
    *[f"| {t} | {m} / {mg} | {l} / {lg} |" for t, m, mg, l, lg in coverage],
    "", "### Held-out by route code", "", "| Code | Cases |", "|---|---|", table(heldout, "service_code"),
    "", "### Held-out by expected outcome", "", "| Outcome | Cases |", "|---|---|", table(heldout, "expected_action"),
    "", "### Held-out by language", "", "| Language | Cases |", "|---|---|", table(heldout, "lang"), "",
]
(OUT / "LOCK.md").write_text("\n".join(lines), encoding="utf-8")

print("LOCKED. All leakage checks passed.")
print(f"  Codes and outcomes from: {policy_source}")
print(f"  Near-identical generated cases dropped: {len(dropped)}")
for n, rows in files.items():
    print(f"  {n}: {len(rows)} cases")
print(f"  Hazard cases in held-out: {len(hz)}")
print(f"  Wrote {OUT / 'LOCK.md'}")
if dropped and len(dropped) > 0.05 * len(test):
    print(f"\nWARNING: {len(dropped)} of {len(test)} generated test cases were near-identical to train. "
          "That is more than 5%, which suggests the split is leaking. Check with Shravan before using these files.")
